# Connection

<div class="alert alert-info">

Namespace `langchain.mcp` yêu cầu `langchain[mcp]>=1.4.0` và đang trong giai đoạn beta. API có thể thay đổi.

</div>

[`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) mở một kết nối MCP, khám phá các tool, và trả về các LangChain tool mà agent của bạn có thể gọi. Cách bạn truyền server vào, và thời gian bạn giữ adapter mở, phụ thuộc vào hình dạng ứng dụng của bạn. Bản thân kết nối thuộc về FastMCP; trang này đề cập đến các pattern của LangChain và liên kết ra ngoài đến FastMCP để biết chi tiết về transport và client.

Ưu tiên [vòng đời kết nối mặc định](https://docs.langchain.com/oss/python/langchain/mcp/connections#connection-lifecycle) trừ khi bạn đang kết nối đến nhiều server hoặc triển khai nhiều lượt chạy đồng thời. Để biết một target đơn lẻ có thể là gì (URL, đường dẫn script, server in-process), xem [Transports](https://docs.langchain.com/oss/python/langchain/mcp#transports).

## Chọn một pattern

Chọn một dòng từ mỗi bảng. Hình dạng server và thời gian sống của kết nối là độc lập với nhau; bất kỳ hình dạng nào cũng hoạt động với bất kỳ thời gian sống nào.

**Hình dạng server**

| Nếu bạn cần…                                                 | Sử dụng                                                          | Đi tới                                                   |
| ------------------------------------------------------------- | ------------------------------------------------------------ | -------------------------------------------------------- |
| Một server                                                    | Một URL, `Path`, hoặc target in-process                      | [Transports](https://docs.langchain.com/oss/python/langchain/mcp#transports)       |
| Nhiều server trong một kết nối                                | Một dict `MCPConfig`                                          | [MCPConfig](https://docs.langchain.com/oss/python/langchain/mcp/connections#one-aggregate-connection-with-mcpconfig)    |
| Nhiều server với xác thực, thế hệ giao thức, hoặc pool riêng | Một [`ClientGroup`](https://gofastmcp.com/clients/client-groups) | [ClientGroup](https://docs.langchain.com/oss/python/langchain/mcp/connections#independent-connections-with-clientgroup) |

**Thời gian sống của kết nối**

| Tình huống                                            | Pattern                                                                                 | Đi tới                                                     |
| ------------------------------------------------------ | ---------------------------------------------------------------------------------------- | ------------------------------------------------------- |
| Script, notebook, hoặc hầu hết các agent                | Khám phá bên trong `async with`, sau đó thoát ra                                          | [Vòng đời kết nối](https://docs.langchain.com/oss/python/langchain/mcp/connections#connection-lifecycle)                |
| Giữ một session xuyên suốt nhiều lần gọi tool trong một lượt chạy | Giữ adapter mở xung quanh lệnh gọi agent                                        | [Một session cho mỗi lần invocation](https://docs.langchain.com/oss/python/langchain/mcp/connections#one-session-per-invocation) |
| Nhiều lượt chạy đồng thời trong một bản triển khai      | Khám phá theo từng lượt chạy; tái sử dụng một [pool dùng chung](https://docs.langchain.com/oss/python/langchain/mcp/connections#shared-connection-pool) và [cache](https://docs.langchain.com/oss/python/langchain/mcp/connections#caching) | [Mở rộng quy mô triển khai](https://docs.langchain.com/oss/python/langchain/mcp/connections#scale-a-deployment)   |

## Vòng đời kết nối

[`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) là một async context manager. Việc bước vào (entering) sẽ kết nối client bên dưới; việc thoát ra (exiting) sẽ giải phóng kết nối. Việc khám phá diễn ra bên trong context, nhưng các tool mà nó trả về vẫn giữ client, nên chúng vẫn có thể được gọi sau khi context kết thúc.

Để khám phá các tool và xây dựng một agent (pattern mặc định):

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter


async def build_agent(target):
    # Khám phá và xây dựng agent bên trong context của adapter. Các tool giữ
    # client, nên agent vẫn có thể sử dụng được sau khi context kết thúc.
    async with MCPAdapter(target) as adapter:
        tools = await adapter.list_tools()
        return create_agent("claude-sonnet-5", tools)

Bạn không cần phải giữ adapter mở trong suốt vòng đời của agent. Ưu tiên sử dụng pattern này trừ khi bạn đang [giữ một session mở](https://docs.langchain.com/oss/python/langchain/mcp/connections#one-session-per-invocation) hoặc [mở rộng quy mô triển khai](https://docs.langchain.com/oss/python/langchain/mcp/connections#scale-a-deployment).

### Một session cho mỗi lần invocation

Các tool mà [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) trả về có tính chất reentrant: mỗi lần một tool được gọi, nó sẽ mở client, thực thi lệnh gọi MCP, và giải phóng nó, bất kể một kết nối đã được giữ ở nơi khác hay chưa. Vì vậy, một lượt chạy agent duy nhất sẽ mở một session cho mỗi lần gọi tool và đóng lại khi lệnh gọi đó hoàn tất, thay vì giữ một session mở xuyên suốt toàn bộ lượt chạy. Điều này giúp một agent chạy trong thời gian dài không giữ một kết nối rảnh rỗi giữa các lần gọi tool, và đó cũng là lý do tại sao các tool vẫn có thể gọi được sau khi context khám phá kết thúc.

Nếu bạn muốn giữ một session mở xuyên suốt nhiều lệnh gọi, hãy giữ context của adapter mở khi bạn gọi agent. Client có tính reentrant sẽ tái sử dụng kết nối hiện có thay vì mở một kết nối thứ hai.

## Nhiều server

Để cung cấp cho một agent các tool từ nhiều server, hãy chọn `MCPConfig` khi một kết nối tổng hợp duy nhất là đủ, hoặc `ClientGroup` khi mỗi server cần kết nối riêng (các [thế hệ giao thức](https://docs.langchain.com/oss/python/langchain/mcp/connections#protocol-eras) khác nhau, [xác thực theo từng server](https://docs.langchain.com/oss/python/langchain/mcp/auth#per-server-authentication), hoặc một [pool dùng chung](https://docs.langchain.com/oss/python/langchain/mcp/connections#shared-connection-pool) được cấu hình riêng cho từng client).

### Một kết nối tổng hợp với `MCPConfig`

Truyền cho adapter một dict `MCPConfig` để kết nối đến nhiều server thông qua một endpoint tổng hợp duy nhất. FastMCP thêm tiền tố vào mỗi tool bằng key cấu hình của nó, nên hai server có cùng tên tool vẫn có thể phân biệt được trong danh sách đưa cho model:

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter

CONFIG = {
    "mcpServers": {
        "weather": {"command": "python", "args": ["/path/to/weather_server.py"]},
        "calc": {"command": "python", "args": ["/path/to/calc_server.py"]},
    }
}


async def fleet_agent(config):
    async with MCPAdapter(config) as adapter:
        # Mỗi tool đều được thêm tiền tố bằng key cấu hình của nó (`weather_...`, `calc_...`),
        # nên hai server có cùng tên tool vẫn có thể phân biệt được.
        tools = await adapter.list_tools()
        return create_agent("claude-sonnet-5", tools)

Mỗi backend được xử lý độc lập, vì vậy một fleet (nhóm server) có thể kết hợp nhiều loại transport: một server qua stdio, một server khác qua HTTP. Tuy nhiên, một fleet `MCPConfig` chia sẻ chung một [thế hệ giao thức](https://docs.langchain.com/oss/python/langchain/mcp/connections#protocol-eras) đã thương lượng trên toàn bộ backend: nếu thêm một server chỉ hỗ trợ giao thức legacy, toàn bộ fleet sẽ bị hạ xuống thế hệ legacy.

### Các kết nối độc lập với `ClientGroup`

Để giữ mỗi server trên một kết nối riêng, hãy truyền vào một [`ClientGroup`](https://gofastmcp.com/clients/client-groups). Mỗi thành viên giữ riêng thế hệ giao thức đã thương lượng, xác thực, và handler của mình, và group sẽ định tuyến mỗi lệnh gọi trở lại đúng client đã công bố tool đó. Đây chính là điều cho phép một server legacy và một server hiện đại chạy song song với nhau, và nó cũng đặt namespace cho các tool theo cùng một cách, nên các tên tool giống nhau trên các server khác nhau sẽ không bao giờ xung đột:

In [ ]:
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter


async def agent_from_group(legacy_url: str, modern_url: str):
    # Một kết nối cho mỗi server: `ClientGroup` giữ riêng cho mỗi server
    # thế hệ giao thức đã thương lượng của nó, nên một server legacy và một
    # server hiện đại có thể chạy song song với nhau.
    # Nó cũng đặt namespace cho mỗi tool theo dạng `{server}_{tool}`, nên hai server
    # có cùng tên tool vẫn tách biệt nhau.
    group = ClientGroup(
        {
            "weather": Client(legacy_url, mode="legacy"),
            "calc": Client(modern_url, mode="auto"),
        }
    )
    async with MCPAdapter(group) as adapter:
        tools = await adapter.list_tools()
        return create_agent("claude-sonnet-5", tools)

## Mở rộng quy mô triển khai

Một bản triển khai phục vụ nhiều lượt chạy nên khám phá theo từng lượt chạy nhưng tái sử dụng kết nối bên dưới, thay vì kết nối lại ở mỗi request. Hãy xây dựng agent bên trong một graph factory của [`langgraph dev`](https://docs.langchain.com/oss/python/langgraph/local-server) để mỗi lượt chạy nắm bắt được danh mục tool hiện tại, và để một [pool kết nối dùng chung](https://docs.langchain.com/oss/python/langchain/mcp/connections#shared-connection-pool) cùng [response cache](https://docs.langchain.com/oss/python/langchain/mcp/connections#caching) hấp thụ chi phí này:

In [ ]:
SERVERS = {
    "weather": "http://localhost:8001/mcp",
    "calc": "http://localhost:8002/mcp",
}


async def make_graph():
    """Xây dựng một agent trên một fleet MCP. Được gọi một lần cho mỗi lượt chạy bởi `langgraph dev`."""
    config = {"mcpServers": {name: {"url": url} for name, url in SERVERS.items()}}
    # Một bản triển khai chạy trong thời gian dài sẽ khám phá theo từng lượt chạy, nhưng tái sử dụng
    # một pool kết nối HTTP bên dưới. `cache_mode="use"` phục vụ một danh sách tool đã cache
    # trong khoảng TTL của server thay vì liệt kê lại ở mỗi lượt chạy.
    async with MCPAdapter(config) as adapter:
        tools = await adapter.list_tools(cache_mode="use")
        return create_agent("claude-sonnet-5", tools)

<div class="alert alert-info">

Trong một graph factory của `langgraph dev`, các kiểu tham số được chú thích và kiểu trả về phải có thể import được tại runtime, không chỉ dưới `TYPE_CHECKING`. `langgraph-api` phân loại factory bằng `typing.get_type_hints()`; nếu một annotation không thể phân giải (resolve) được, nó sẽ chèn vào một config dict thay vì runtime.

</div>

Để xem một ví dụ triển khai đầy đủ, bao gồm xác thực theo từng người dùng để cấp (mint) một token cho mỗi bên gọi, xem [Authentication](https://docs.langchain.com/oss/python/langchain/mcp/auth#per-user-authentication).

### Pool kết nối dùng chung

Theo mặc định, mỗi FastMCP client tự quản lý các kết nối HTTP của riêng nó. Trên một fleet server, hoặc nhiều lượt chạy đồng thời, điều đó có nghĩa là có nhiều pool độc lập. Để chia sẻ một pool duy nhất, hãy truyền vào một `httpx_client_factory` lấy dữ liệu từ một transport duy nhất, và cho các client mượn transport đó mà không để bất kỳ client nào đóng nó lại:

In [ ]:
import httpx2
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup
from fastmcp.client.transports import StreamableHttpTransport
from langchain.mcp import MCPAdapter

# Một pool kết nối duy nhất, được chia sẻ bởi mọi server mà bản triển khai giao tiếp.
_POOL = httpx2.AsyncHTTPTransport()


class _SharedPool(httpx2.AsyncBaseTransport):
    """Cho mỗi client mượn `_POOL` mà không để client nào đóng nó lại."""

    handle_async_request = _POOL.handle_async_request

    async def aclose(self) -> None: ...


def _client_factory(**kwargs: object) -> httpx2.AsyncClient:
    return httpx2.AsyncClient(transport=_SharedPool(), **kwargs)


async def load_over_shared_pool(servers: dict[str, str]) -> list:
    # Mỗi client lấy kết nối HTTP từ cùng một pool, nên một fleet server sẽ
    # không mỗi server tự mở pool riêng của mình.
    group = ClientGroup(
        {
            name: Client(
                StreamableHttpTransport(url, httpx_client_factory=_client_factory)
            )
            for name, url in servers.items()
        }
    )
    async with MCPAdapter(group) as adapter:
        return await adapter.list_tools()

Vì mỗi client đều mượn từ `_POOL`, bản triển khai chỉ mở một tập hợp kết nối HTTP cho toàn bộ fleet thay vì mở riêng cho từng server.

### Caching

FastMCP có thể cache kết quả của `list_tools` để việc khám phá lặp lại tránh được một round trip mạng. Caching là tùy chọn (opt-in) và tôn trọng các gợi ý cache (cache hint) riêng của server, nên nó chỉ có hiệu lực đối với các server thế hệ hiện đại có công bố các gợi ý đó.

`list_tools()` nhận một tham số `cache_mode` để chọn cách khám phá đọc từ một cache đã được cấu hình:

* **`use`** (mặc định): phục vụ một danh sách tool đã cache khi có sẵn và vẫn còn trong khoảng TTL hint của server, nếu không thì fetch và lưu lại.
* **`refresh`**: fetch một danh sách mới từ server và điền lại vào cache.
* **`bypass`**: bỏ qua cache hoàn toàn.

In [ ]:
tools = await adapter.list_tools(cache_mode="refresh")

Cache và cơ chế cô lập theo từng principal của nó được cấu hình ngay trên client, bằng `Client(cache=...)`. Để biết cách dùng một store dùng chung cho một fleet các bản sao (replica), hoặc phân vùng (partition) các response đã cache theo từng người dùng, xem [Response caching](https://gofastmcp.com/clients/client#response-caching) trong tài liệu của FastMCP.

## Các thế hệ giao thức

MCP đã thay đổi cách một client và server thỏa thuận về những gì mỗi bên hỗ trợ. Thế hệ **legacy** bắt đầu mỗi kết nối bằng một cú bắt tay (handshake) `initialize`; thế hệ **hiện đại** (phiên bản giao thức `2026-07-28` trở lên) khám phá khả năng hỗ trợ bằng cách probe endpoint `server/discover` của server. FastMCP thương lượng thế hệ giao thức theo từng kết nối, nên phía LangChain không cần phải biết một server cụ thể nào đó đang dùng thế hệ nào.

Để giữ các tool từ các server thuộc các thế hệ khác nhau trong cùng một agent, hãy cho mỗi server một kết nối riêng để nó giữ được thế hệ tốt nhất mà server của nó hỗ trợ, thông qua một [`ClientGroup`](https://docs.langchain.com/oss/python/langchain/mcp/connections#independent-connections-with-clientgroup) hoặc một adapter riêng cho mỗi server. Một `fastmcp.Client` dựng sẵn sẽ chọn thế hệ giao thức bằng tham số `mode` của nó:

In [ ]:
from fastmcp.client import Client


async def agent_across_eras(legacy_target, modern_target):
    # MCP có hai thế hệ giao thức. FastMCP thương lượng theo từng kết nối, nên
    # một adapter riêng cho mỗi server cho phép mỗi server giữ được thế hệ
    # tốt nhất mà chính server đó hỗ trợ. `mode="legacy"` cố định thế hệ handshake;
    # `mode="auto"` (mặc định) thương lượng thế hệ mới nhất mà server hiểu được.
    legacy = Client(legacy_target, mode="legacy")
    modern = Client(modern_target, mode="auto")
    async with (
        MCPAdapter(legacy) as legacy_adapter,
        MCPAdapter(modern) as modern_adapter,
    ):
        tools = await legacy_adapter.list_tools() + await modern_adapter.list_tools()
        return create_agent("claude-sonnet-5", tools)

Nếu thay vào đó truyền cả hai server dưới dạng một fleet `MCPConfig` duy nhất, hệ thống sẽ thương lượng một thế hệ giao thức chung cho toàn bộ fleet, kéo mọi server xuống thế hệ cũ nhất mà bất kỳ thành viên nào yêu cầu.

Để biết đầy đủ các quy tắc thương lượng, xem [Protocol negotiation](https://gofastmcp.com/clients/client#protocol-negotiation) trong tài liệu của FastMCP.

## Xem thêm

* [Authentication](https://docs.langchain.com/oss/python/langchain/mcp/auth) — bearer, OAuth, và thông tin xác thực theo từng người dùng
* [Deploy a LangGraph server](https://docs.langchain.com/oss/python/langgraph/local-server) — các graph factory cho các bản triển khai chạy dài hạn
* [FastMCP connection lifecycle](https://gofastmcp.com/clients/client#connection-lifecycle)
* [MCP configuration format](https://gofastmcp.com/integrations/mcp-json-configuration)